# Random Forest

In [3]:
import os
import cv2
import numpy as np
import pandas as pd
import time
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder

In [4]:
from google.colab import drive
drive.mount('/content/data', force_remount=True)

ROOT_DIRECTORY = 'data/Shareddrives/ai4museums/projetos/tipografia-MuseuPaulista'
SPLITS_CSV_PATH = ROOT_DIRECTORY + '/k-fold_split/typography_dataset_splits.csv'

Mounted at /content/data


In [5]:
def load_fold_from_csv(csv_path, target_fold_idx):
    """
    Reads the global splits CSV and reconstructs the train/val DataFrames for a specific fold.
    """
    df = pd.read_csv(csv_path)
    
    # Filter by the specific fold number
    fold_df = df[df['fold'] == target_fold_idx]
    
    # Separate into train and validation
    train_df = fold_df[fold_df['split_type'] == 'train']
    val_df = fold_df[fold_df['split_type'] == 'val']
    
    return train_df, val_df

In [6]:
def extract_classical_features(dataframe, target_size=(64, 64)):
    features = []
    labels = []
    
    for _, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc="Extraindo imagens"):
        img_path = row['image_path']
        label = row['label']
        
        # Load image in grayscale (reduces 3 channels to 1)
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            continue
            
        # Resize to standard size (handles varied dimensions)
        img_resized = cv2.resize(img, target_size)
        
        # Flatten into 1D vector
        flattened_features = img_resized.flatten()
        
        features.append(flattened_features)
        labels.append(label)
        
    return np.array(features), np.array(labels)

In [ ]:
def train_evaluate_random_forest(csv_path, label_encoder, num_folds=5, image_size=(64, 64)):
    print(f"Starting Random Forest evaluation with image size {image_size}")
    fold_metrics = []
    
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=2)
    
    # Agora iteramos sobre o número de folds em vez de uma variável na memória
    for fold_idx in range(num_folds):
        print(f"\n{'='*40}")
        print(f"Processing Fold {fold_idx + 1}/{num_folds}")
        print(f"{'='*40}")
        
        # Lendo os DataFrames diretamente do CSV
        train_df, val_df = load_fold_from_csv(csv_path, target_fold_idx=fold_idx)
        
        # Extração de características (usando a sua função existente)
        X_train, y_train_text = extract_classical_features(train_df, target_size=image_size)
        X_val, y_val_text = extract_classical_features(val_df, target_size=image_size)
        
        y_train = label_encoder.transform(y_train_text)
        y_val = label_encoder.transform(y_val_text)
        
        print("Training Random Forest...")
        start_time = time.time()
        rf_classifier.fit(X_train, y_train)
        training_time = time.time() - start_time
        
        print("Evaluating...")
        y_pred = rf_classifier.predict(X_val)
        
        fold_accuracy = accuracy_score(y_val, y_pred)
        print(f"Fold {fold_idx + 1} Accuracy: {fold_accuracy:.4f} (Training took {training_time:.2f}s)")
        
        target_names = label_encoder.classes_
        print(classification_report(y_val, y_pred, labels=[0, 1, 2, 3], target_names=target_names, zero_division=0))
        
        fold_metrics.append(fold_accuracy)
        
    avg_accuracy = np.mean(fold_metrics)
    std_accuracy = np.std(fold_metrics)
    
    print(f"\n{'='*40}")
    print("FINAL RANDOM FOREST CROSS-VALIDATION RESULTS")
    print(f"Average Accuracy: {avg_accuracy:.4f} ± {std_accuracy:.4f}")
    print(f"{'='*40}\n")
    
    return rf_classifier, fold_metrics

In [8]:
if __name__ == "__main__":
    
    if os.path.exists(SPLITS_CSV_PATH):
        classes = ['grotesco', 'serifado', 'escritural', 'fantasia']
        label_encoder = LabelEncoder()
        label_encoder.fit(classes)
        
        # Passamos o caminho do CSV e a quantidade de folds para a função
        trained_rf, rf_accuracies = train_evaluate_random_forest(
            csv_path=SPLITS_CSV_PATH, 
            label_encoder=label_encoder,
            num_folds=5,
            image_size=(64, 64)
        )
    else:
        print(f"Erro: O arquivo '{SPLITS_CSV_PATH}' não foi encontrado.")

Starting Random Forest evaluation with image size (64, 64) and SMOTE...

Processing Fold 1/5


Extraindo imagens: 100%|██████████| 212/212 [02:13<00:00,  1.59it/s]
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.


Training Random Forest...
building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100

[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:    1.0s


building tree 43 of 100
building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82 of 100
building tree 83 of 100
building tree 84

...

Extraindo imagens: 100%|██████████| 239/239 [00:01<00:00, 217.82it/s]
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.


Training Random Forest...
building tree 1 of 100
building tree 2 of 100
building tree 3 of 100
building tree 4 of 100
building tree 5 of 100
building tree 6 of 100
building tree 7 of 100
building tree 8 of 100
building tree 9 of 100
building tree 10 of 100
building tree 11 of 100
building tree 12 of 100
building tree 13 of 100
building tree 14 of 100
building tree 15 of 100
building tree 16 of 100
building tree 17 of 100
building tree 18 of 100
building tree 19 of 100
building tree 20 of 100
building tree 21 of 100
building tree 22 of 100
building tree 23 of 100
building tree 24 of 100
building tree 25 of 100
building tree 26 of 100
building tree 27 of 100
building tree 28 of 100
building tree 29 of 100
building tree 30 of 100
building tree 31 of 100
building tree 32 of 100
building tree 33 of 100
building tree 34 of 100
building tree 35 of 100
building tree 36 of 100
building tree 37 of 100
building tree 38 of 100
building tree 39 of 100
building tree 40 of 100
building tree 41 of 100

[Parallel(n_jobs=-1)]: Done  37 tasks      | elapsed:    1.6s


building tree 44 of 100
building tree 45 of 100
building tree 46 of 100
building tree 47 of 100
building tree 48 of 100
building tree 49 of 100
building tree 50 of 100
building tree 51 of 100
building tree 52 of 100
building tree 53 of 100
building tree 54 of 100
building tree 55 of 100
building tree 56 of 100
building tree 57 of 100
building tree 58 of 100
building tree 59 of 100
building tree 60 of 100
building tree 61 of 100
building tree 62 of 100
building tree 63 of 100
building tree 64 of 100
building tree 65 of 100
building tree 66 of 100
building tree 67 of 100
building tree 68 of 100
building tree 69 of 100
building tree 70 of 100
building tree 71 of 100
building tree 72 of 100
building tree 73 of 100
building tree 74 of 100
building tree 75 of 100
building tree 76 of 100
building tree 77 of 100
building tree 78 of 100
building tree 79 of 100
building tree 80 of 100
building tree 81 of 100
building tree 82 of 100
building tree 83 of 100
building tree 84 of 100
building tree 85

[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    3.2s finished
[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  37 tasks      | elapsed:    0.0s
[Parallel(n_jobs=2)]: Done 100 out of 100 | elapsed:    0.0s finished
